# 03 Docker与部署 · 第01课：Docker与容器基础

> 本课从你已经完成的FastAPI聊天后端出发，解决“为什么换一台电脑后还要重新准备Python、依赖和启动方式”的问题。我们先掌握镜像、容器、生命周期和端口映射，不提前进入Dockerfile、数据卷或Compose。

## 1. 与上一阶段的连接和本课目标

第02阶段的AI聊天后端在本机虚拟环境中已经通过34项测试。虚拟环境能够隔离Python包，却没有规定另一台机器的操作系统环境、系统库、启动命令和端口。Docker要解决的是更完整的交付一致性问题。

学完本课后，你应能独立：

1. 解释宿主机、Docker Engine、镜像、容器和镜像仓库；
2. 区分镜像与容器，说明一个镜像为什么能创建多个容器；
3. 看懂`docker pull`、`run`、`ps`、`logs`、`stop`、`start`和`rm`；
4. 解释容器主进程与容器生命周期的关系；
5. 正确解释`宿主机端口:容器端口`；
6. 根据容器状态、日志和端口映射做基础排错。

## 2. 运行本课前：Docker环境

本课的Python小练习和作业只需要`pytest`，没有Docker也能学习并运行自动检测。标有“真实Docker实践”的命令必须等Docker Engine可用后再执行。

当前这台设备是Apple Silicon Mac，应安装适用于Apple silicon的Docker Desktop。Docker官方当前说明Docker Desktop支持当前及之前两个主要macOS版本，并要求至少4 GB内存。安装入口和要求以[Docker Desktop for Mac官方文档](https://docs.docker.com/desktop/setup/install/mac-install/)为准。

安装后在终端检查：

```powershell
docker version
docker run --rm hello-world
```

`docker version`中：`docker`是命令行客户端；`version`是子命令；成功时结果应同时包含Client和Server信息。只有Client通常表示命令已安装但Docker Engine尚未启动。

本课作业依赖安装：

```powershell
cd "03_Docker与部署/01_Docker与容器基础"
python -m pip install -r requirements.txt
```

`python -m pip`让当前Python运行其对应的pip；`install`表示安装；`-r requirements.txt`读取同目录依赖清单。本课只有`pytest`，用于运行作业测试。

## 3. 为什么虚拟环境还不够

虚拟环境主要隔离Python解释器可见的第三方包。它通常不包含操作系统、系统动态库、端口规则，也不会自动告诉别人如何启动服务。

| 问题 | 虚拟环境 | 容器 |
|---|---|---|
| 隔离Python包 | 是 | 是，可以装在镜像中 |
| 固定基础操作系统环境 | 否 | 是，镜像声明用户空间环境 |
| 统一启动命令 | 不自动提供 | 镜像可以定义默认主进程 |
| 隔离进程和网络视图 | 否 | 是，但与虚拟机隔离方式不同 |
| 跨机器交付 | 仍需重新配环境 | 交付镜像和运行配置 |

容器不是完整虚拟机。它通常共享宿主机内核，通过操作系统提供的隔离机制运行进程，因此启动更快、开销更小，但也不应把容器误解成绝对安全边界。

## 4. Docker整体模型

先记住一条主线：

```text
Docker CLI发送命令
        ↓
Docker Engine执行拉取、创建和运行
        ↓
镜像（只读模板）→ 创建 → 容器（运行实例）
        ↑
镜像仓库负责保存和分发镜像
```

- **宿主机**：真正运行Docker的电脑；
- **Docker CLI**：你在终端输入的`docker`命令；
- **Docker Engine**：接收命令并管理镜像、容器和网络的后台服务；
- **镜像**：创建容器需要的只读模板，包含文件和默认运行配置；
- **容器**：由某个镜像创建出来的实例，可以处于created、running或exited等状态；
- **镜像仓库**：保存与分发镜像的服务，例如Docker Hub。

可以暂时用“类与实例”帮助理解镜像和容器，但这只是类比：Python实例是内存中的对象，容器则是受隔离的进程及其文件系统、网络等运行配置。

### 课堂小练习1：区分镜像与容器

**目标**：判断描述对应镜像还是容器。

请把下方`object_types`三个值补成`"image"`或`"container"`。预期全部正确时检测输出✅。

In [ ]:
object_types = {
    "nginx:alpine这个可重复使用的模板": "TODO",
    "名为web-1且正在运行的实例": "TODO",
    "停止后仍可再次start的实例": "TODO",
}

In [ ]:
# 课堂小练习1自动检测
_check_expected = ["image", "container", "container"]
try:
    _check_actual = list(object_types.values())
    if _check_actual == _check_expected:
        print("✅ 全部正确：已经能区分镜像与容器。")
    else:
        print(f"❌ 当前答案为{_check_actual}，请重新对照模板与运行实例。")
except NameError:
    print("❌ 没有找到object_types，请先运行上方答题单元。")

<details><summary>完成后查看参考答案</summary>

依次是`image`、`container`、`container`。停止只改变容器状态，不会自动把容器变回镜像。

</details>

## 5. 镜像名称、标签与拉取

常见镜像引用写成`仓库名:标签`，例如`nginx:alpine`。`nginx`是仓库名，`alpine`是标签。标签由镜像发布者定义，常用于区分版本或变体；它不是Python关键字，也不等同于容器名称。

```powershell
docker pull nginx:alpine
docker image ls
```

- `docker pull IMAGE`：从仓库拉取镜像；`IMAGE`是位置参数；成功后镜像保存在本机；
- `docker image ls`：列出本机镜像；`image`是管理镜像的命令组，`ls`是列出子命令。

如果省略标签，Docker通常使用`latest`，但`latest`只是普通标签，不保证它永远代表最稳定版本。学习和项目配置中优先明确标签。

## 6. 容器生命周期与常用命令

`docker run`不是单纯的“运行已有容器”，它会基于镜像创建一个新容器并启动。停止后的同一个容器应使用`docker start`再次启动。

```text
镜像 --docker run--> 新容器（running）
                         |
                   docker stop
                         ↓
                    容器（exited）
                         |
                   docker start
                         ↓
                    再次running

停止容器 --docker rm--> 容器被删除
```

| 命令 | 关键参数 | 作用 |
|---|---|---|
| `docker run IMAGE` | 镜像引用 | 创建并启动新容器 |
| `docker ps` | 无 | 列出运行中的容器 |
| `docker ps -a` | `-a`：全部 | 同时显示停止的容器 |
| `docker logs NAME` | 容器名或ID | 读取容器标准输出和错误输出 |
| `docker stop NAME` | 容器名或ID | 请求容器停止 |
| `docker start NAME` | 已停止容器 | 再次启动同一容器 |
| `docker rm NAME` | 已停止容器 | 删除容器，不删除镜像 |

Docker官方将`docker container run`描述为“从镜像创建并运行新容器”，短写形式是`docker run`。`docker rm`负责删除容器。

### 课堂小练习2：选择生命周期命令

**目标**：根据目标选择命令。请填写子命令名称，不写`docker`前缀。

In [ ]:
lifecycle_commands = {
    "查看包括已停止容器在内的全部容器": "TODO",
    "停止名为web-1的容器": "TODO",
    "重新启动同一个web-1容器": "TODO",
    "删除已经停止的web-1容器": "TODO",
}

In [ ]:
# 课堂小练习2自动检测
_check_expected = ["ps -a", "stop web-1", "start web-1", "rm web-1"]
try:
    _check_actual = list(lifecycle_commands.values())
    if _check_actual == _check_expected:
        print("✅ 全部正确：容器生命周期命令选择通过。")
    else:
        print(f"❌ 当前答案为{_check_actual}，请检查run与start、stop与rm的区别。")
except NameError:
    print("❌ 没有找到lifecycle_commands，请先运行答题单元。")

<details><summary>完成后查看参考答案</summary>

依次是`ps -a`、`stop web-1`、`start web-1`、`rm web-1`。`run`会创建新容器，`start`才是启动已有容器。

</details>

## 7. 主进程为什么决定容器是否存活

容器启动时会运行一个主进程。主进程持续运行，容器通常保持running；主进程结束，容器就进入exited。

正常示例：Web服务器以前台方式持续监听请求，容器保持运行。变化示例：一个打印一句话就结束的程序会让容器正常退出。错误示例：尝试把服务放到后台后，启动脚本本身立即结束，Docker看到主进程结束，容器也会停止。

`docker run -d IMAGE`中的`-d`是`--detach`的短写，表示让容器在后台运行，不占用当前终端。它不会让已经结束的主进程凭空继续存活。Docker官方文档也强调，后台容器会在其根进程退出时停止。

排查“容器一启动就退出”时：

1. `docker ps -a`确认状态；
2. `docker logs 容器名`查看程序输出；
3. 检查启动命令是否报错或立即执行完毕。

## 8. 端口映射：两个端口分别属于谁

容器拥有自己的网络视图。程序在容器内监听8000端口，不代表宿主机的8000端口已经自动连接过去。运行时使用`-p`或`--publish`发布端口：

```powershell
docker run --name web-1 -d -p 8080:80 nginx:alpine
```

读取规则始终是：

```text
-p 宿主机端口:容器端口
   8080       80
```

浏览器访问`http://127.0.0.1:8080`，Docker把请求转发到容器的80端口。根据Docker官方端口发布文档，`-p 8080:80`建立宿主机8080到容器80的映射。

常见错误：

- 写反两侧端口；
- 程序实际监听8000，却映射到容器80；
- 完全没有`-p`，然后期待从宿主机直接访问；
- 宿主机端口已经被另一个程序占用。

发布到所有宿主机地址可能让局域网其他设备访问该端口。学习阶段若只需本机访问，可以进一步指定`127.0.0.1:8080:80`；是否对外开放还受操作系统防火墙和网络环境影响。

下面的Python函数只是帮助练习解析，不会执行Docker：`split(":", maxsplit=1)`来自字符串类型，用冒号最多切分一次并返回字符串列表；`int()`来自Python内置函数，把数字字符串转换成整数。

In [ ]:
def explain_port_mapping(mapping: str) -> dict[str, int]:
    host_text, container_text = mapping.split(":", maxsplit=1)
    return {
        "host_port": int(host_text),
        "container_port": int(container_text),
    }

print(explain_port_mapping("8080:80"))

### 课堂小练习3：读取端口映射

**目标**：明确访问端口和应用监听端口。对于`-p 9000:8000`，填写两个整数变量。

In [ ]:
browser_port = 0       # 浏览器在宿主机访问的端口
application_port = 0   # 应用在容器内监听的端口

In [ ]:
# 课堂小练习3自动检测
try:
    if not isinstance(browser_port, int) or not isinstance(application_port, int):
        print("❌ 两个答案都应是整数。")
    elif (browser_port, application_port) == (9000, 8000):
        print("✅ 正确：浏览器访问9000，容器应用监听8000。")
    else:
        print(f"❌ 当前为{browser_port}:{application_port}，请按宿主机:容器读取。")
except NameError:
    print("❌ 请先定义browser_port和application_port。")

<details><summary>完成后查看参考答案</summary>

`browser_port = 9000`，`application_port = 8000`。左边属于宿主机，右边属于容器。

</details>

## 9. 拆解一条`docker run`命令

```powershell
docker run -d --name web-1 -p 8080:80 nginx:alpine
```

| 部分 | 类型 | 含义 |
|---|---|---|
| `docker` | 命令 | 调用Docker CLI |
| `run` | 子命令 | 创建并启动新容器 |
| `-d` | 布尔选项 | 后台运行；出现表示True，省略表示False |
| `--name web-1` | 带值选项 | 将容器命名为`web-1` |
| `-p 8080:80` | 带值选项 | 发布宿主机8080到容器80 |
| `nginx:alpine` | 位置参数 | 用于创建容器的镜像 |

选项通常放在镜像名之前。镜像名之后的内容可能被当作容器内要执行的命令或参数，因此随意改变顺序会改变含义。

`--rm`表示容器退出后自动删除，适合一次性练习。它与`docker rm`不同：前者在创建容器时设置自动清理行为，后者显式删除一个已有的停止容器。

### 课堂小练习4：组合参数列表

**目标**：理解命令、选项、选项值和镜像位置参数的顺序。将`command_parts`补成等价于`docker run --rm --name demo -p 9000:80 nginx:alpine`的字符串列表。

In [ ]:
command_parts = ["docker", "run", "TODO"]

In [ ]:
# 课堂小练习4自动检测
_check_expected = ["docker", "run", "--rm", "--name", "demo", "-p", "9000:80", "nginx:alpine"]
try:
    if not isinstance(command_parts, list):
        print("❌ command_parts必须是列表。")
    elif command_parts == _check_expected:
        print("✅ 正确：docker run参数顺序通过。")
    else:
        print(f"❌ 当前列表为{command_parts}，请检查选项值和镜像位置。")
except NameError:
    print("❌ 没有找到command_parts，请先运行答题单元。")

<details><summary>完成后查看参考答案</summary>

`["docker", "run", "--rm", "--name", "demo", "-p", "9000:80", "nginx:alpine"]`。列表形式也正是Python命令执行器常用的安全参数形式，但本课不实际执行它。

</details>

## 10. 基础排错：先确定问题在哪一层

遇到“打不开页面”时，不要立刻重装Docker。按顺序缩小范围：

1. `docker version`：客户端能否连接Engine；
2. `docker image ls`：需要的镜像是否存在；
3. `docker ps -a`：容器是running还是exited；
4. `docker logs 容器名`：主进程是否报错；
5. `docker port 容器名`或`docker ps`的PORTS列：映射是否正确；
6. 检查浏览器访问的是宿主机端口，而不是容器端口。

| 现象 | 优先检查 | 可能原因 |
|---|---|---|
| 只有Client，没有Server | Docker Engine | Docker Desktop未启动 |
| 容器状态Exited | `docker logs` | 主进程完成或启动失败 |
| 容器Running但访问失败 | 端口映射 | 漏写`-p`或两侧端口不匹配 |
| 提示容器名已存在 | `docker ps -a` | 同名停止容器尚未删除 |
| 提示端口已被占用 | 宿主机端口 | 另一个进程或容器已占用 |

`docker logs`读取的是容器主进程写入标准输出和标准错误的内容；它不会自动理解你的业务逻辑，但能提供启动错误和运行日志。

### 课堂小练习5：选择第一条排错命令

**目标**：针对现象选择最直接的检查。填写完整命令。

In [ ]:
first_checks = {
    "确认Docker客户端能否连接Engine": "TODO",
    "容器web-1一启动就退出，查看原因": "TODO",
    "查看包括停止容器在内的状态": "TODO",
}

In [ ]:
# 课堂小练习5自动检测
_check_expected = ["docker version", "docker logs web-1", "docker ps -a"]
try:
    _check_actual = list(first_checks.values())
    if _check_actual == _check_expected:
        print("✅ 正确：基础排错入口选择通过。")
    else:
        print(f"❌ 当前答案为{_check_actual}，请按Engine、日志、状态三层重新判断。")
except NameError:
    print("❌ 没有找到first_checks，请先运行答题单元。")

<details><summary>完成后查看参考答案</summary>

依次为`docker version`、`docker logs web-1`、`docker ps -a`。先收集证据，再决定是否修改配置。

</details>

## 11. 真实Docker实践（安装后执行）

以下命令使用官方`nginx:alpine`镜像，不涉及本课作业答案：

```powershell
docker pull nginx:alpine
docker run -d --name lesson-nginx -p 8080:80 nginx:alpine
docker ps
docker logs lesson-nginx
docker stop lesson-nginx
docker ps -a
docker start lesson-nginx
docker stop lesson-nginx
docker rm lesson-nginx
```

启动后访问`http://127.0.0.1:8080`。练习目标是亲眼观察：镜像不会因为容器删除而消失；停止的容器可以再次启动；同一镜像可以创建多个不同名称的容器。

清理时不使用`docker system prune`等大范围命令，因为它可能删除其他项目尚未使用的资源。只删除本课明确创建的`lesson-nginx`。

## 12. 完整执行流程回顾

```text
1. 用户输入docker run及选项
2. Docker CLI把请求发送给Docker Engine
3. Engine检查本机是否已有镜像，没有则尝试从仓库拉取
4. Engine根据镜像创建一个新容器及其运行配置
5. Engine启动容器主进程
6. 如果配置了-p，Engine建立宿主机到容器端口的转发
7. 主进程持续运行时容器保持running
8. 主进程结束或用户stop后，容器进入exited
9. start可重启停止的容器；rm删除容器但不删除镜像
```

哪些是你做的：输入命令、选择镜像、名称和端口。哪些是CLI做的：解析命令并请求Engine。哪些是Engine做的：拉取、创建、隔离、启动和维护状态。哪些是容器内程序做的：监听端口、处理请求、输出日志。

## 13. 课末作业：容器启动命令生成器

### 任务背景

在自动化工具中，通常先把外部命令组织成字符串列表，再交给命令执行器。你要完成`容器命令练习.py`，把本课的镜像引用、端口映射和`docker run`参数规则表达成可测试的Python函数。本作业只生成数据，不执行Docker，因此测试不会下载镜像或启动容器。

### 允许修改范围

只修改`容器命令练习.py`中的三个TODO函数；不要修改`test_容器命令练习.py`降低要求。

### 函数契约

#### `make_image_reference(repository: str, tag: str) -> str`

- 去除两个输入的首尾空白；
- 任一部分为空时抛出`ValueError`；
- 返回`repository:tag`格式字符串。

#### `parse_port_mapping(mapping: str) -> tuple[int, int]`

- 输入格式必须是`宿主机端口:容器端口`，并且只能有一个冒号；
- 两侧必须是十进制整数；
- 两个端口都必须在1到65535之间；
- 成功时返回`(宿主机端口, 容器端口)`；非法输入抛出`ValueError`。

#### `build_run_command(...) -> list[str]`

- `image`、`container_name`去除首尾空白后不能为空；
- 两个端口必须在1到65535之间；
- 固定开头为`["docker", "run"]`；
- `detached=True`时加入`-d`，默认值就是True；
- `auto_remove=True`时加入`--rm`，默认值是False；
- 随后依次加入`--name`、容器名、`-p`、端口映射；镜像必须放在最后；
- 非法名称或端口抛出`ValueError`；
- 只能返回参数列表，不能调用`subprocess`或执行真实命令。

`detached`和`auto_remove`是仅限关键字参数，因为函数签名中的`*`要求调用者写出参数名，避免多个布尔值仅凭位置难以阅读。

### 测试

```powershell
python -m pytest test_容器命令练习.py -v
```

应收集并通过**24项测试**。测试覆盖正常镜像引用、首尾空白、空名称、正常与边界端口、非法映射、两个布尔选项组合、命令顺序以及禁止执行真实Docker。

## 14. 理解题

请用自己的话回答：

1. Python虚拟环境与Docker容器分别主要隔离什么？
2. 镜像和容器是什么关系？停止容器后镜像会发生什么？
3. `docker run`与`docker start`有什么区别？
4. 为什么容器主进程结束后，容器通常也会停止？`-d`能否改变这一点？
5. `-p 8080:80`两侧端口分别属于谁？浏览器应该访问哪一个？
6. 容器显示running但页面打不开时，为什么应先检查端口映射，而不是直接重装Docker？
7. `--rm`和`docker rm`有什么区别？为什么本课清理时不使用大范围`docker system prune`？

### 在这里填写理解题答案

**1. 虚拟环境与容器**

（请填写。）

**2. 镜像与容器**

（请填写。）

**3. run与start**

（请填写。）

**4. 主进程与-d**

（请填写。）

**5. 端口映射**

（请填写。）

**6. 分层排错**

（请填写。）

**7. 容器清理**

（请填写。）

## 15. 本课小结与下一课

本课建立了Docker的运行模型：CLI请求Engine，Engine从镜像创建容器，容器主进程决定生命周期，端口映射把宿主机请求送入容器。你还学会了用状态、日志和端口逐层排错。

下一课`02_Dockerfile与镜像构建`将回答：镜像内部的Python、依赖和文件从哪里来，以及如何用Dockerfile可重复地构建自己的镜像。

官方参考：

- [Docker Desktop for Mac安装](https://docs.docker.com/desktop/setup/install/mac-install/)
- [`docker container run`](https://docs.docker.com/reference/cli/docker/container/run/)
- [端口发布与映射](https://docs.docker.com/engine/network/port-publishing/)
- [`docker container`命令组](https://docs.docker.com/reference/cli/docker/container/)